# 09 · 注意力机制与 Transformer

> Transformer 是当今大模型（GPT / BERT / ViT）的基础。
> 核心思想：**让每个位置直接"关注"所有其它位置**，取代循环。

## 9.1 从注意力说起

注意力 = 加权求和。给定：
- **Query (Q)**：我要找什么
- **Key (K)**：我能提供什么
- **Value (V)**：我实际的内容

$$
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

- $QK^\top$ 计算相似度分数
- $\sqrt{d_k}$ 缩放，防止点积过大导致 softmax 梯度消失
- softmax 归一化成权重
- 加权求和 V

## 9.2 缩放点积注意力（手写实现）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def scaled_dot_product_attention(q, k, v, mask=None):
    # q: (..., Tq, d_k)  k: (..., Tk, d_k)  v: (..., Tk, d_v)
    # mask: 可广播到 (..., Tq, Tk)，True/1 表示保留，False/0 表示屏蔽
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)     # (..., Tq, Tk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    out = attn @ v                                        # (..., Tq, d_v)
    return out, attn


torch.manual_seed(0)
B, T, d = 2, 4, 8
q = torch.randn(B, T, d)
k = torch.randn(B, T, d)
v = torch.randn(B, T, d)
out, attn = scaled_dot_product_attention(q, k, v)
print("输出:", out.shape, " 注意力权重:", attn.shape)
print("每行权重和为 1:", attn.sum(dim=-1)[0])

In [ ]:
# 可视化注意力权重
import matplotlib.pyplot as plt
plt.figure(figsize=(4, 3.5))
plt.imshow(attn[0].detach(), cmap="viridis")
plt.colorbar(); plt.xlabel("Key 位置"); plt.ylabel("Query 位置")
plt.title("注意力权重矩阵 (batch 0)")
plt.show()

## 9.3 Padding Mask

变长序列里，padding 位置不应被关注。构造 mask：

In [ ]:
lengths = torch.tensor([4, 2])
Tk = 4
mask = torch.arange(Tk)[None, :] < lengths[:, None]     # (B, Tk)
print("mask:\n", mask.int())

# 扩展成 (B, 1, Tk) 以便广播到 (B, Tq, Tk)
mask2 = mask[:, None, :]
out, attn = scaled_dot_product_attention(q, k, v, mask=mask2)
print("\n第 1 个样本只有前 2 个位置有权重（长度 2）:")
print(attn[1])

## 9.4 多头注意力（Multi-Head Attention）

一个注意力头只能学一种"关注模式"。多头注意力：
1. 把 d_model 切成 h 个 d_head = d_model / h
2. 每个头独立做注意力（不同的 Q/K/V 投影）
3. 拼接后线性变换

$$
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)W^O
$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.h = num_heads
        self.d_head = d_model // num_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B, Tq, _ = q.shape
        Tk = k.size(1)

        def split(x):
            return x.view(B, -1, self.h, self.d_head).transpose(1, 2)   # (B,h,T,dh)

        qh, kh, vh = split(self.Wq(q)), split(self.Wk(k)), split(self.Wv(v))
        out, attn = scaled_dot_product_attention(qh, kh, vh, mask)      # (B,h,Tq,dh)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.Wo(out), attn


mha = MultiHeadAttention(32, num_heads=4)
x = torch.randn(2, 6, 32)
out, attn = mha(x, x, x)          # 自注意力
print("输出:", out.shape, " 权重:", attn.shape)   # (2,4,6,6)
print("参数量:", sum(p.numel() for p in mha.parameters()))

In [ ]:
# 与官方实现对比
official = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
x = torch.randn(2, 6, 32)
out_off, attn_off = official(x, x, x, need_weights=True)
print("官方 MultiheadAttention 输出:", out_off.shape, " 权重:", attn_off.shape)

## 9.5 位置编码（Positional Encoding）

注意力本身**没有顺序概念**（置换等变）。必须显式加入位置信息。

经典正弦编码：
$$
PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right),\quad
PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))     # (1, max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


pe = PositionalEncoding(64, max_len=50)
x = torch.zeros(1, 50, 64)
out = pe(x)
plt.figure(figsize=(10, 4))
plt.imshow(out[0].detach().T, aspect="auto", cmap="RdBu")
plt.xlabel("位置 pos"); plt.ylabel("维度 d"); plt.title("正弦位置编码")
plt.colorbar(); plt.show()

> 现代模型常用**可学习位置编码**（BERT）或 **RoPE 旋转位置编码**（LLaMA）。

## 9.6 完整 Transformer Block

一个 block = 多头注意力 + 前馈网络，都带**残差连接**和 **LayerNorm**：

```
x = x + Attention(LN(x))        # Pre-LN（现代推荐）
x = x + FFN(LN(x))
```
或 Post-LN（原始论文）：
```
x = LN(x + Attention(x))
x = LN(x + FFN(x))
```

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        h = self.ln1(x)                         # Pre-LN
        a, _ = self.attn(h, h, h, mask)
        x = x + self.dropout(a)                 # 残差
        x = x + self.ffn(self.ln2(x))           # 残差
        return x


blk = TransformerBlock(64, 4)
x = torch.randn(2, 10, 64)
print("Block 输出:", blk(x).shape)

## 9.7 用 nn.TransformerEncoder 搭建分类器

PyTorch 内置 `nn.TransformerEncoderLayer` / `nn.TransformerEncoder`。

任务：序列分类（例如判断序列是否递增）。

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=1, d_model=64, num_heads=4,
                 num_layers=2, num_classes=2, max_len=50):
        super().__init__()
        self.embed = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len, dropout=0.1)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=4 * d_model,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, num_classes))

    def forward(self, x, src_key_padding_mask=None):
        h = self.pos(self.embed(x))                        # (B,T,d_model)
        h = self.encoder(h, src_key_padding_mask=src_key_padding_mask)
        h = h.mean(dim=1)                                  # 平均池化
        return self.head(h)


model = TransformerClassifier()
x = torch.randn(4, 20, 1)
print("输出:", model(x).shape)
print("参数量:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 造数据：判断序列是否递增
def make_order_data(n=1000, T=20, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, T, 1, generator=g)
    y = (X[:, 1:, 0] > X[:, :-1, 0]).all(dim=1).long()      # 全部递增=1
    return X, y

X, y = make_order_data()
print("正样本比例:", y.float().mean().item())

from torch.utils.data import TensorDataset, DataLoader
n_tr = 800
tr_loader = DataLoader(TensorDataset(X[:n_tr], y[:n_tr]), batch_size=64, shuffle=True)
te_loader = DataLoader(TensorDataset(X[n_tr:], y[n_tr:]), batch_size=128)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerClassifier().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
crit = nn.CrossEntropyLoss()

for epoch in range(1, 16):
    model.train()
    tl, corr, n = 0, 0, 0
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item() * len(yb); corr += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    model.eval()
    tc, tn = 0, 0
    with torch.no_grad():
        for xb, yb in te_loader:
            xb, yb = xb.to(device), yb.to(device)
            tc += (model(xb).argmax(1) == yb).sum().item(); tn += len(yb)
    if epoch % 3 == 0:
        print(f"epoch {epoch:2d} | train_loss {tl/n:.4f} acc {corr/n:.3f} | test acc {tc/tn:.3f}")

## 9.8 因果掩码（Causal Mask）与自回归

GPT 类模型生成时，位置 $t$ 只能看到 $\le t$ 的位置。用一个上三角掩码：

In [ ]:
T = 5
causal = torch.tril(torch.ones(T, T)).bool()
print("因果掩码 (True=可见):\n", causal.int())

x = torch.randn(1, T, 8)
out, attn = scaled_dot_product_attention(x, x, x, mask=causal)
print("\n注意力权重（下三角，未来不可见）:\n", attn[0].round(decimals=2))

## 9.9 完整 Transformer 模型（编码器-解码器）

```python
model = nn.Transformer(d_model=64, nhead=4, num_encoder_layers=2,
                       num_decoder_layers=2, batch_first=True)
```
用于机器翻译等 seq2seq 任务。这里演示调用形状。

In [ ]:
transformer = nn.Transformer(d_model=32, nhead=4,
                             num_encoder_layers=2, num_decoder_layers=2,
                             dim_feedforward=64, batch_first=True)
src = torch.randn(2, 10, 32)      # 源序列
tgt = torch.randn(2, 7, 32)       # 目标序列（右移后）
out = transformer(src, tgt)
print("nn.Transformer 输出:", out.shape)    # (2, 7, 32)
print("参数量:", f"{sum(p.numel() for p in transformer.parameters()):,}")

## 9.10 ViT 思路：把图像当序列

把图像切成小块（patch），展平成序列送入 Transformer：
```
(C,H,W) -> 切成 P×P 的 patch -> (N_patches, P*P*C) -> Linear -> Transformer
```

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch=4, in_ch=3, d_model=64):
        super().__init__()
        self.n_patches = (img_size // patch) ** 2
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch, stride=patch)

    def forward(self, x):
        x = self.proj(x)                             # (B, d_model, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)             # (B, N, d_model)
        return x

pe = PatchEmbedding()
img = torch.randn(4, 3, 32, 32)
tokens = pe(img)
print("patch 数量:", pe.n_patches, " token 形状:", tokens.shape)

# 加一个可学习的 cls token
cls = nn.Parameter(torch.randn(1, 1, 64))
tokens = torch.cat([cls.expand(4, -1, -1), tokens], dim=1)
print("加 cls token 后:", tokens.shape)

## 9.11 Transformer vs RNN

| 维度 | RNN | Transformer |
|---|---|---|
| 并行性 | 差（必须按时间步） | 好（全并行） |
| 长程依赖 | 难（梯度衰减） | 强（任意两位置直连） |
| 复杂度 | O(T·d²) | O(T²·d) |
| 短序列 | 有优势 | 也可 |
| 超长序列 | O(T) 线性 | O(T²) 爆炸 → 需稀疏/线性注意力 |

`nn.TransformerEncoderLayer` 还支持 `src_key_padding_mask`、
`enable_nested_tensor` 等优化。

## 9.12 练习

**练习 1**：修改 `MultiHeadAttention`，让 `num_heads=1`，验证它退化成单头注意力。

**练习 2**：把 `TransformerClassifier` 的平均池化换成"取 [CLS] token"，
在输入前加一个可学习 token，比较效果。

**练习 3**：在序列分类任务里去掉位置编码，看准确率是否下降（体会位置信息的重要性）。

**练习 4**：把 `norm_first` 从 True 改成 False（Post-LN），比较收敛。

**练习 5**：写一个自回归 mini GPT：给定随机整数序列，预测下一个 token，
用因果掩码 + `CrossEntropyLoss`。

In [ ]:
# 练习 5：mini GPT 自回归
class MiniGPT(nn.Module):
    def __init__(self, vocab=20, d_model=64, heads=4, layers=2, max_len=32):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, heads, 4 * d_model,
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok(idx) + self.pos(pos)
        mask = torch.triu(torch.ones(T, T, device=idx.device), 1).bool()  # True=屏蔽
        x = self.enc(x, mask=mask)
        return self.head(x)

# 造一个简单的确定性序列：0..19 循环
V = 20
data = torch.arange(0, V).repeat(50).reshape(50, V)
X = data[:, :-1]                    # (50, 19)
Y = data[:, 1:]                     # 预测下一个
gpt = MiniGPT(vocab=V)
opt = torch.optim.AdamW(gpt.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
for step in range(200):
    opt.zero_grad()
    logits = gpt(X)
    loss = crit(logits.reshape(-1, V), Y.reshape(-1))
    loss.backward(); opt.step()
    if (step + 1) % 50 == 0:
        acc = (gpt(X).argmax(-1) == Y).float().mean().item()
        print(f"练习5 step {step+1} | loss {loss.item():.4f} | acc {acc:.3f}")

## 9.13 小结

- 注意力：`softmax(QKᵀ/√d)V`，多头 = 多个子空间并行关注。
- 位置编码弥补注意力缺失的顺序信息。
- Transformer Block = MultiHeadAttention + FFN + 残差 + LayerNorm。
- 掩码：padding mask（屏蔽补齐）、causal mask（屏蔽未来）。
- ViT 把图像切 patch 当 token；GPT 用因果掩码自回归。

下一节：`10_迁移学习与预训练模型.ipynb`